## Purpose
This script performs all of our webscrapping and saves the relevant data into json files. Lets explain the purpose of FBR and FBW:
* FBRfExtractor(FBR): This class handles the extraction and scraping of data from FBRef. (the precise logic will be commented in the fbref.py file)
  
* FBRefDataWriter (FBW): Following the data extraction from FBRef, we need to save this data, so this classhandles the writing of data to the write directories (within the data directory and the resepective season for the data we have extracted)

Psuedo logic for code:
1. iterate over each specified season
2. extract all teams player in this season
3. extract all players, and their info, for all teams in this season
4. then get the fixtures for this league
5. if they've already occured, get all match logs (telling us about the performance for each player that featured in that game)

Although AI was not used directly in any of the code here, it was used to help create the all_players_dict cache logic.

In [6]:
from utilities.fbref import FBRefExtractor, FBRefDataWriter
import os

# first we need to locate the data directory, which is just a directory below our current working directory
data_directory = os.path.join(os.getcwd(),'data', "seasons")
# initialise both of these clases, note that FBW needs the data directory location to funtion
FBR = FBRefExtractor()
FBW = FBRefDataWriter(data_directory)

# declare the seasons that we wish to scrape for. Change if you wish
premier_league_seasons = ["2021-2022", "2022-2023", "2023-2024", "2024-2025"]

# this dictionary is used as a cache for players when scraping multiple seasons. In a nutshell, after we scrape and extract data for a player from their page, 
# we store their details. So in the case when they appear across multiple seasons we don't need to rescrape their page, we can simply copy across their data
# from the other season they featured in . this helps reduce run time as MANY players appear in MANY seasons
all_players_dict = dict()

# now lets begin scraping for each season within the declared scope
for season in premier_league_seasons:

    # here we need to set the season for FBR to scrape.  THis is because we construct endpoints within the class and the endpoints are constructed differently
    # depending on if the season is the current season
    FBR.set_season(season)

    # initialise the dictionary for storing all players playing in THIS season
    # key: Player Name, value: Player Details
    season_all_players = dict()
    # this dictionary will store all players for each team in this season
    # key: Team Name, value: list of player names(str)
    team_players_dict = dict()

    # to extract team link, we first need to get the season endpoint of FBRef, so we construct it, and then parse it to extract_team_links to get a list of all team endpoints
    # FBR.extract_team_links returns a list of tuples of size 2(team_name, team_page (dependent on season) endpoint)
    team_links = FBR.extract_team_links(FBR.construct_season_endpoint())
    
    # we're using enumerate here so we can do logging.
    for team_count, team_data in enumerate(team_links.items()):
        

        # unpack the tuple of data as described before
        team_name, team_link = team_data
        # create a list of players playing in the team, to be added as the value of key team in the team_players_dict
        team_players_list = list()

        # extract all player links from the teams page. This returns a dictionary
        # key: Player name, value: tuple of size 2: players name and their endpoint
        player_links = FBR.extract_player_links(team_link)

        # using enumerate here as well to do logging. We are now going to iterate over the dictionry
        for player_count,  player_data in enumerate(player_links.items()):
            player_name, player_endpoint = player_data #unpack data
            print(f"  Player {player_count + 1}/{len(player_links)}: {player_name}")
            
            team_players_list.append(player_name) #add the players name to the team_players_list

            # if we've encountered this player previously in another season, just retrieve that player info and add it to the season players dict
            if player_name in all_players_dict:
                season_all_players[player_name] = all_players_dict[player_name]
            # if not, lets extract the details
            else:
                # FBR EXTRACT_PLAYER_details returns a dictionary of their details
                player_data = FBR.extract_player_details(player_endpoint)
                season_all_players[player_name] = player_data

        # after we've iteraeted over all players in a team, add the team players list to the dictionary
        team_players_dict[team_name] = team_players_list
        print(f"Finished team: {team_name}")
    # after iterating over all teams, if we've encountered new players, add them to the cache for future usage
    all_players_dict = all_players_dict | season_all_players
    print("Finished processing teams.")

    # write this seasons player and team details to json files using FBW
    for data, filename in [(season_all_players, "players"),(team_players_dict, "teams")]:
        FBW.write_data(data, season, filename)
    print("All seasons processed. Saved to File.")
    
    # for this season extract the fixtures (match table). this returns some html. we need to first construct the fixtures endpoint
    fixtures = FBR.get_fixtures()
    # lets turn this html table in to a dictionary, and do some formatting
    fixtures_dict = FBR.fixtures_to_dict(fixtures)
    # for each match, if possible, add match data information. this is data about each players performance within that match
    fixtures_dict = FBR.append_match_data(fixtures_dict)

    # write all this data to a fixtures.json
    FBW.write_data(fixtures_dict,season, "fixtures_new")

  Player 1/33: Ederson
  Player 2/33: João Cancelo
  Player 3/33: Rodri
  Player 4/33: Bernardo Silva
  Player 5/33: Aymeric Laporte
  Player 6/33: Rúben Dias
  Player 7/33: Kevin De Bruyne
  Player 8/33: Phil Foden
  Player 9/33: Raheem Sterling
  Player 10/33: Jack Grealish
  Player 11/33: Gabriel Jesus
  Player 12/33: İlkay Gündoğan


KeyboardInterrupt: 

In [10]:
for season in premier_league_seasons:
    print(season)
    # here we need to set the season for FBR to scrape.  THis is because we construct endpoints within the class and the endpoints are constructed differently
    # depending on if the season is the current season
    FBR.set_season(season)
# for this season extract the fixtures (match table). this returns some html. we need to first construct the fixtures endpoint
    fixtures = FBR.get_fixtures()
    # lets turn this html table in to a dictionary, and do some formatting
    fixtures_dict = FBR.fixtures_to_dict(fixtures)
    # for each match, if possible, add match data information. this is data about each players performance within that match
    fixtures_dict = FBR.append_match_data(fixtures_dict)

    # write all this data to a fixtures.json
    FBW.write_data(fixtures_dict,season, "fixtures_new")

2021-2022
2022-2023
2023-2024


ConnectionError: HTTPSConnectionPool(host='fbref.com', port=443): Max retries exceeded with url: /en/matches/b84d060a/Wolverhampton-Wanderers-Newcastle-United-October-28-2023-Premier-League (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f9a21cdc910>: Failed to establish a new connection: [Errno 51] Network is unreachable'))